# **Hill Country Revenue Forecast**: Regression — HF Deployment
---

## **Value Proposition**

In order to predict weekly product-store revenue across a regional Texas grocery chain, a gradient boosting regression model was developed that achieves R² = 0.8994 on the held-out test set, enabling merchandising teams to forecast per-item weekly revenue before placing orders and setting shelf allocations. Accurate weekly forecasts reduce over-ordering by flagging low-revenue items early and surface which promo configurations lift revenue, directly informing markdown and inventory decisions.

### Business Opportunities

* **Opportunity A:** Store managers currently rely on prior-week actuals when setting shelf allocations and ordering quantities. Weekly revenue per item-store combination varies significantly across departments and store formats.
* **Solution A:** The model predicts `Weekly_Revenue_USD` per item × store combination one week ahead using pricing, packaging, and store attributes. It achieves a mean absolute error of $12.30 on the held-out test set, keeping prediction error within one standard deviation of typical weekly fluctuations for most product lines.

* **Opportunity B:** Promotion pricing (`Promo_Price_USD` vs `List_Price_USD`) is a direct lever the chain controls weekly. Without a predictive model, the revenue impact of a 10% discount is estimated from historical averages that don't account for item or store context.
* **Solution B:** The deployed `/predict` endpoint accepts any promo price as input, so category managers can compare predicted revenue at list price versus promo price for any item-store pair before committing to a markdown.

* **Opportunity C:** The forecast is deployed as a live API on Hugging Face Spaces, with a Streamlit frontend for direct use by non-technical staff. An Administrator section provides model governance and bulk-inference for backtests and ordering-list scoring.
* **Solution C** (Options): Single-item point prediction with 95% confidence interval. Multi-item basket total across any store combination. Bulk batch scoring via CSV upload in the Administrator section.

### Outcomes

* **Model Performance**
  * Primary: **HistGradientBoostingRegressor** — Test R² = 0.8994, MAPE = 15.2%, MAE = $12.30
  * Secondary: **XGBRegressor** — Test R² = 0.8631, MAPE = 17.7%, MAE = $14.34
  * Both calibrated on the same 1,776-row held-out calibration set; primary selected by test R²

* **Architecture**
  * Gated pipeline harness sweeps six candidates; top-two by cross-validation R² are tuned via GridSearchCV — validated by [Notebook A (leak-detection)](https://evagaiml.github.io/000-gated-pipeline-cases/leak-detection__regression__gated-harness/)
  * Split-conformal prediction intervals: shared 1776-row calibration set, asymmetric residual quantiles (alpha=0.05)

* **Economic Impact**
  * A MAE of $12.30/week per item-store row translates to ordering decisions made within roughly one standard error of the actual week's revenue for most SKUs
  * The promo lever in the prediction form quantifies discount ROI before a markdown decision is made, avoiding revenue dilution from unnecessary promos on items that would have sold at list price

* **Strategy Recommendation**
  * **Enterprise:** Integrate `/predict_batch` into the weekly replenishment workflow via API call from the ERP; use the Administrator model selector to govern which model serves production and run backtests before switching
  * **SMB:** Use the Streamlit UI for weekly planning meetings; category managers input the upcoming week's planned promotions and read the predicted revenue impact per item before finalizing the flyer

## **Problem Space**

### Overview

* Weekly revenue per item-store combination drives ordering quantities, shelf allocation, and promo calendar decisions across the Hill Country Grocer chain. Getting it wrong in either direction costs the chain: over-forecast leads to excess inventory and markdowns; under-forecast leads to stockouts and lost sales.
* The chain spans 6 stores across multiple Texas regions. Revenue varies by department, store size, and promo status — a model that ignores any of these axes will produce systematically biased forecasts for the stores or categories it under-represents.
* `Weekly_Revenue_USD` is the product of units sold and effective price. The dataset does not carry a temporal anchor (no week-of-year column), so seasonal trends are not directly modeled here.

### Data Description

* **8880** weekly item × store observations total: **5328** training rows, **1776** calibration rows, and **1776** test rows
* 12 predictive features after dropping `Item_Description` (display label) and the `UPC` identifier: 5 categorical and 7 numeric
* Data source: `hill_country_grocer_weekly_sales.csv`, loaded via the shared Layer-2 data loader
* Target variable: `Weekly_Revenue_USD` — continuous, no zeros
* Secondary target: `Weekly_Units_Sold` — excluded as co-target per the gated pipeline's leakage rules

### Process

* Three-way split (60% / 20% / 20%) enforced before any model fitting; the 20% calibration set is held out entirely for split-conformal CI computation and never seen during model selection or tuning.
* Gated pipeline harness sweeps six candidates on 3-fold cross-validation of the training set; top-two by CV R² proceed to GridSearchCV tuning and test evaluation.
* Conformal quantiles computed independently for each model on the shared held-out calibration set using signed residuals; these scalars shift every prediction to produce the 95% interval.

### Results
>| Model | Test R² | Role |
>|---|---|---|
>| | **HistGradientBoostingRegressor ✅** | 0.8994 | Lead deployment model |
>| | XGBRegressor | 0.8631 | Challenger / governance fallback |

# **Code Execution**

### **Runtime Configuration**

> **Hardware Accelerator:** **CPU** and **Standard RAM**
>
> This notebook trains two gradient boosting models on tabular data and deploys to Hugging Face Spaces. No GPU is required.

* Installing Colab Dependencies may require a Runtime restart. You will be notified if a restart is required.
* `HF_TOKEN` must be set in Colab Secrets before running the deployment cells.
* gated_pipeline is cloned from `https://github.com/EvagAIML/gated-pipeline.git` if not found locally.

### Install Dependencies

**Process:** Install training, data-loading, and HF deployment packages. Uses a flag file to skip reinstall after runtime restart.

**Outcome:** Environment ready.

In [1]:
import os, sys
from pathlib import Path
from IPython.display import display, HTML

_in_colab = os.path.exists("/content")
flag_file = Path("/content/.deps_installed_flag") if _in_colab else Path(".deps_installed_flag")

if not flag_file.exists():
    print("Installing dependencies...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "scikit-learn==1.6.1", "xgboost>=1.7",
        "joblib", "pandas", "numpy", "pyarrow",
        "huggingface_hub>=0.25", "requests",
    ], check=True)
    flag_file.touch()
    if _in_colab:
        display(HTML("""
        <div style="border:4px solid #d93025;background:#fce8e6;color:#202124;padding:24px;
                    border-radius:12px;font-family:Arial,sans-serif;margin:16px 0;">
            <div style="font-size:30px;font-weight:800;color:#b31412;margin-bottom:12px;">
                ⚠️ Runtime Restart Required
            </div>
            <div style="font-size:22px;line-height:1.5;font-weight:600;">
                Dependencies installed. Go to <b>Runtime → Restart session and run all</b>.
            </div>
        </div>
        """))
    else:
        print("Dependencies installed.")
else:
    print("✅ Dependencies already installed.")

✅ Dependencies already installed.


### Library Import and Configuration

**Process:** Import required libraries and the gated_pipeline harness. In Colab, the harness is cloned from `https://github.com/EvagAIML/gated-pipeline.git`.

**Outcome:** All libraries and harness functions imported; `SEED = 42` set for reproducibility.

In [2]:
# ------------------------------
# LIBRARY IMPORT AND CONFIGURATION
# ------------------------------
# Standard ML libraries + gated_pipeline harness (cloned from GitHub for Colab).
import copy, os, sys, json, warnings, hashlib, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import pyarrow

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
from huggingface_hub import HfApi

# gated_pipeline: clone from GitHub in Colab, use local path otherwise
_gp_local = "/Users/eriksvagshenian/Documents/Claude Cowork/Tasks/gated-pipeline"
_gp_colab = "/content/gated-pipeline"
_in_colab = os.path.exists("/content")

if _in_colab:
    import subprocess
    if not os.path.exists(_gp_colab):
        subprocess.run(["git", "clone", "https://github.com/EvagAIML/gated-pipeline.git", _gp_colab], check=True)
    sys.path.insert(0, _gp_colab)
else:
    sys.path.insert(0, _gp_local)

# XGBoost 1.7.x compat patch for sklearn 1.6+: add __sklearn_tags__ to XGBModel
# so _find_tags_provider returns "__sklearn_tags__" (path with try/except fallback).
try:
    from xgboost.sklearn import XGBModel as _XGBModel
    from sklearn.base import BaseEstimator as _BE, RegressorMixin as _RM
    if "__sklearn_tags__" not in _XGBModel.__dict__:
        def _xgb_sklearn_tags(self):
            return super(_XGBModel, self).__sklearn_tags__()
        _XGBModel.__sklearn_tags__ = _xgb_sklearn_tags
except Exception:
    pass

import gated_pipeline as gp

# Manual KFold CV: avoids cross_val_score for XGBoost 1.7.x sklearn 1.6+ compat
def manual_cv_r2(model, X, y, n_splits=3, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = []
    for ti, vi in kf.split(X):
        m = copy.deepcopy(model)
        m.fit(X[ti], y.iloc[ti])
        scores.append(float(r2_score(y.iloc[vi], m.predict(X[vi]))))
    return float(np.mean(scores)), float(np.std(scores))

SEED   = gp.SEED
TARGET = "Weekly_Revenue_USD"
CO_TARGET    = "Weekly_Units_Sold"
DISPLAY_ONLY = ["Item_Description"]

print("sklearn:", __import__("sklearn").__version__)
print("xgboost:", __import__("xgboost").__version__)
print("gated_pipeline.SEED:", gp.SEED)

sklearn: 1.6.1
xgboost: 1.7.6
gated_pipeline.SEED: 42


### HF Authentication

**Process:** Read `HF_TOKEN` from Colab Secrets or environment. Confirm the authenticated HF username.

**Outcome:** `api` and `HF_OWNER` ready; Space repo IDs and backend URL set.

In [3]:
# ------------------------------
# HF AUTHENTICATION
# ------------------------------
# Colab: read from Secrets. Local: read from .env file or environment.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

if not os.environ.get("HF_TOKEN"):
    _env_candidates = [
        pathlib.Path("/Users/eriksvagshenian/Documents/Claude Cowork/.env"),
        pathlib.Path.home() / "Documents" / "Claude Cowork" / ".env",
    ]
    for _ep in _env_candidates:
        if _ep.exists():
            for _line in _ep.read_text().splitlines():
                _line = _line.strip()
                if _line.startswith("HF_TOKEN=") and not _line.startswith("#"):
                    _val = _line.split("=", 1)[1].strip().strip('"').strip("'")
                    if _val:
                        os.environ["HF_TOKEN"] = _val
                        break
            if os.environ.get("HF_TOKEN"):
                break

if not os.environ.get("HF_TOKEN"):
    raise EnvironmentError(
        "HF_TOKEN not set. In Colab: key icon → +Add new secret → name=HF_TOKEN, Notebook access ON."
    )

api = HfApi()
me  = api.whoami()
HF_OWNER = me["name"]
print(f"Authenticated as: {HF_OWNER}")
BACKEND_REPO  = f"{HF_OWNER}/hill-country-revenue-forecast-backend"
FRONTEND_REPO = f"{HF_OWNER}/hill-country-revenue-forecast-frontend"
BACKEND_URL   = f"https://{HF_OWNER.lower()}-hill-country-revenue-forecast-backend.hf.space"
print(f"Backend:  {BACKEND_REPO}")
print(f"Frontend: {FRONTEND_REPO}")

Authenticated as: EvagAIML
Backend:  EvagAIML/hill-country-revenue-forecast-backend
Frontend: EvagAIML/hill-country-revenue-forecast-frontend


### Data Loading

**Process:** Load `hill_country_grocer_weekly_sales.csv` (8880 rows) via the shared Layer-2 data loader.

**Outcome:** `df` ready. Target has no zeros, making MAPE valid.

In [4]:
# ------------------------------
# DATA LOADING
# ------------------------------
import importlib.util

_shared_path = pathlib.Path("_shared")
if not _shared_path.exists():
    _shared_path = pathlib.Path("/content/notebooks/_shared")
spec = importlib.util.spec_from_file_location("data_access", _shared_path / "data_access.py")
data_access = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_access)

data_dir = data_access.ensure_dataset("hill-country-revenue-forecast__regression__hf-deployment")
# The zip extracts to data_dir/raw/; check both locations for compatibility
csv_path = data_dir / "raw" / "hill_country_grocer_weekly_sales.csv"
if not csv_path.exists():
    csv_path = data_dir / "hill_country_grocer_weekly_sales.csv"
df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df[TARGET].describe().round(2)}")

Dataset shape: (8880, 16)
Target stats:
count    8880.00
mean       83.29
std        63.31
min         6.99
25%        42.29
50%        66.46
75%       104.00
max       748.21
Name: Weekly_Revenue_USD, dtype: float64


### Data Overview and Feature Selection

**Process:** Drop `Item_Description` (display label), `Weekly_Units_Sold` (co-target), and any identifier columns (nunique/len > 0.9). Confirm the final 12-feature set.

**Outcome:** `X` has 12 features (5 categorical, 7 numeric). Target zeros: 0.

In [5]:
# ------------------------------
# DATA OVERVIEW AND FEATURE SELECTION
# ------------------------------
y = df[TARGET].copy()
X = df.drop(columns=[TARGET, CO_TARGET] + DISPLAY_ONLY)

id_cols = [c for c in X.columns if X[c].nunique() / len(X) > 0.9]
print(f"Identifier columns dropped: {id_cols}")
if id_cols:
    X = X.drop(columns=id_cols)

cats = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
nums = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
print(f"Features: {len(X.columns)} ({len(cats)} categorical, {len(nums)} numeric)")
print(f"Categorical: {cats}")
print(f"Numeric:     {nums}")
print(f"Target zeros: {(y == 0).sum()}")

Identifier columns dropped: ['UPC']
Features: 12 (5 categorical, 7 numeric)
Categorical: ['Department', 'Brand_Type', 'Store_Number', 'Store_Banner', 'Store_Region']
Numeric:     ['Net_Weight_oz', 'Pack_Size', 'List_Price_USD', 'Promo_Price_USD', 'Shelf_Facings', 'Store_Sq_Ft', 'Store_Open_Year']
Target zeros: 0


### Exploratory Data Analysis

**Process:** Compute department revenue shares and per-store row counts to characterize dataset composition before splitting.

**Outcome:** Department revenue shares and store distributions printed for reference.

In [6]:
# ------------------------------
# EXPLORATORY DATA ANALYSIS
# ------------------------------
dept_rev   = df.groupby("Department")[TARGET].sum().sort_values(ascending=False)
dept_share = (dept_rev / dept_rev.sum() * 100).round(1)
print("Department revenue share (%):")
print(dept_share.to_string())
print("\nRows per store:")
print(df["Store_Number"].value_counts().sort_index().to_string())

Department revenue share (%):
Department
Household                13.5
Meat & Seafood           12.3
Dairy                    12.1
Produce                  11.2
Beverages                10.3
Center Store - Snacks     8.1
Center Store - Pantry     7.0
Frozen                    6.7
Deli                      6.4
Bakery                    4.3
Breakfast & Cereal        4.1
Health & Beauty           4.0

Rows per store:
Store_Number
HCG-101    1480
HCG-102    1480
HCG-103    1480
HCG-104    1480
HCG-105    1480
HCG-106    1480


### Three-Way Split and OHE Preprocessing

**Process:** Partition the dataset into three non-overlapping sets: 5328 training rows (60%), 1776 calibration rows (20%), 1776 test rows (20%). The calibration set is held out entirely from model selection and tuning — it is only used after both winning models are chosen to compute conformal quantiles.

**Analysis:** Split-conformal prediction requires a calibration set exchangeable with the test set but unused for model selection or tuning. The OHE is fit on X_train only — no test or calibration rows inform the encoder fit.

**Outcome:** Three disjoint sets established. OHE output dimension confirms categorical encoding.

In [7]:
# ------------------------------
# THREE-WAY SPLIT AND OHE PREPROCESSING
# ------------------------------
# 60% train · 20% calibration (held out for conformal CI) · 20% test.
# OHE fit on training rows only — calibration and test never touch the encoder fit.
X_temp, X_test,  y_temp, y_test  = train_test_split(X, y, test_size=0.20, random_state=SEED)
X_train, X_cal,  y_train, y_cal  = train_test_split(X_temp, y_temp, test_size=0.25, random_state=SEED)
print(f"train={len(X_train)}  cal={len(X_cal)}  test={len(X_test)}")

ohe = ColumnTransformer([
    ("c", OneHotEncoder(handle_unknown="ignore", min_frequency=10,
                        max_categories=20, sparse_output=False), cats),
    ("n", "passthrough", nums),
])
ohe.fit(X_train)
Xtr_t  = ohe.transform(X_train)
Xcal_t = ohe.transform(X_cal)
Xte_t  = ohe.transform(X_test)
feature_names = ohe.get_feature_names_out().tolist()
print(f"OHE output dim: {len(feature_names)}")

train=5328  cal=1776  test=1776


OHE output dim: 34


### Model Selection — Gated Pipeline 6-Candidate Sweep

**Process:** Evaluate six candidates on 3-fold cross-validation of X_train. X_cal is NOT touched. The two highest CV R² candidates proceed to tuning.

**Analysis:** Selection leaderboard (CV R² mean on X_train):

```
  1. XGBoost      : CV R²=0.8799
  2. HistGBR      : CV R²=0.8784
  3. ExtraTrees   : CV R²=0.8328
  4. RandomForest : CV R²=0.8301
  5. Ridge        : CV R²=0.6905
  6. DecisionTree : CV R²=0.6727
```

Top-2 for tuning: ['XGBoost', 'HistGBR'].

**Outcome:** `top2_names` set. No test or calibration data was used.

In [8]:
# ------------------------------
# MODEL SELECTION — 6-CANDIDATE HARNESS SWEEP (3-FOLD CV ON X_TRAIN)
# ------------------------------
# X_cal is NOT touched here. Candidate ranking is on X_train 3-fold CV only.
# Uses manual KFold CV to avoid XGBoost 1.7.x / sklearn 1.6+ __sklearn_tags__ compat issue.
candidates = {
    "RandomForest": gp.RandomForestRegressor(n_estimators=80, random_state=SEED, n_jobs=-1),
    "ExtraTrees":   gp.ExtraTreesRegressor(n_estimators=80, random_state=SEED, n_jobs=-1),
    "HistGBR":      gp.HistGradientBoostingRegressor(random_state=SEED),
    "DecisionTree": gp.DecisionTreeRegressor(random_state=SEED),
    "Ridge":        gp.Ridge(),
    "XGBoost":      gp.XGBRegressor(n_estimators=150, random_state=SEED, n_jobs=-1, verbosity=0),
}
selection_board = []
for name, m in candidates.items():
    mr, sr = manual_cv_r2(m, Xtr_t, y_train, n_splits=3, seed=SEED)
    selection_board.append((name, mr, sr))
    print(f"  {name:13s}: mean_R²={mr:.4f}  std={sr:.4f}")

selection_board.sort(key=lambda t: -t[1])
print("\nSelection leaderboard:")
for rank, (name, mr, sr) in enumerate(selection_board, 1):
    print(f"  {rank}. {name:13s}: {mr:.4f} ± {sr:.4f}")

top2_names = [selection_board[0][0], selection_board[1][0]]
print(f"\nTop-2 for tuning: {top2_names}")

  RandomForest : mean_R²=0.8301  std=0.0068


  ExtraTrees   : mean_R²=0.8328  std=0.0102


  HistGBR      : mean_R²=0.8784  std=0.0055
  DecisionTree : mean_R²=0.6727  std=0.0397


  Ridge        : mean_R²=0.6905  std=0.0087


  XGBoost      : mean_R²=0.8799  std=0.0105

Selection leaderboard:
  1. XGBoost      : 0.8799 ± 0.0105
  2. HistGBR      : 0.8784 ± 0.0055
  3. ExtraTrees   : 0.8328 ± 0.0102
  4. RandomForest : 0.8301 ± 0.0068
  5. Ridge        : 0.6905 ± 0.0087
  6. DecisionTree : 0.6727 ± 0.0397

Top-2 for tuning: ['XGBoost', 'HistGBR']


### Hyperparameter Tuning — Top-2 Candidates

**Process:** Run 3-fold GridSearchCV for each top-2 candidate on X_train. The harness backtrack contract (up to 2 grid expansions) ensures the best params are interior to the search range. After tuning, each model is evaluated on X_test to determine the primary/secondary assignment.

**Analysis:** Primary (HistGradientBoostingRegressor) best params: {'max_depth': 12}, test R²=0.8994. Secondary (XGBRegressor) best params: {'max_depth': 9}, test R²=0.8631. Primary selected by test R² (0.8994 > 0.8631). Tie-break: none.

**Outcome:** `primary_entry` and `secondary_entry` set with trained models and test metrics.

In [9]:
# ------------------------------
# HYPERPARAMETER TUNING — TOP-2 CANDIDATES
# ------------------------------
# GridSearchCV + harness backtrack contract (up to 2 expansions).
# X_cal is NOT touched here; tuning uses X_train via gp.stage_tune.
P_tune = dict(Xtr=X_train, ytr=y_train, ohe=ohe)
candidate_results = []

for name in top2_names:
    print(f"\nTuning {name}...")
    base, grid, exp_param = gp.TUNE_REGISTRY[name]
    grid = dict(grid)
    gs, edge = gp.stage_tune(P_tune, base, grid, attempt=1)
    bt, prev_cv = 0, float(gs.best_score_)
    while edge and bt < 2:
        grid     = gp.expand_grid(grid, exp_param)
        gs, edge = gp.stage_tune(P_tune, base, grid, attempt=bt + 2)
        new_cv   = float(gs.best_score_)
        if new_cv - prev_cv < 0.005:
            break
        prev_cv = new_cv
        bt += 1
    tuned     = gs.best_estimator_
    y_te_pred = tuned.predict(Xte_t)
    test_r2   = float(r2_score(y_test, y_te_pred))
    test_mape = float(mean_absolute_percentage_error(y_test, y_te_pred) * 100)
    test_mae  = float(mean_absolute_error(y_test, y_te_pred))
    print(f"  {name}: R²={test_r2:.4f}  MAPE={test_mape:.2f}%  MAE=${test_mae:.2f}")

    perm   = permutation_importance(tuned, Xte_t, y_test, n_repeats=3, random_state=SEED, n_jobs=-1)
    fi_top = sorted(zip(feature_names, perm.importances_mean), key=lambda t: -t[1])[:5]
    top5_fi = [{"name": str(f), "importance": round(float(i), 4)} for f, i in fi_top]

    hyperparams = {k: (int(v) if isinstance(v, (int,)) else float(v) if isinstance(v, float) else v)
                   for k, v in gs.best_params_.items()}
    candidate_results.append({
        "model_family_short": name,
        "model_family": type(tuned).__name__,
        "model": tuned,
        "test_r2": test_r2, "test_mape_pct": test_mape, "test_mae": test_mae,
        "hyperparams": hyperparams, "top5_feature_importances": top5_fi,
    })

# Sort by locked rule: primary = highest test R²; secondary = runner-up
candidate_results.sort(key=lambda r: (-r["test_r2"], r["test_mape_pct"], r["model_family"]))
primary_entry, secondary_entry = candidate_results[0], candidate_results[1]
print(f"\nPrimary:   {primary_entry['model_family']}  R²={primary_entry['test_r2']:.4f}")
print(f"Secondary: {secondary_entry['model_family']}  R²={secondary_entry['test_r2']:.4f}")


Tuning XGBoost...


  [HYPERTUNE  ] attempt 1  {'model': 'XGBRegressor', 'best': {'max_depth': 3}, 'cv_r2': np.float64(0.8868), 'grid_edge': ['max_depth']}


  [HYPERTUNE  ] attempt 2  {'model': 'XGBRegressor', 'best': {'max_depth': 9}, 'cv_r2': np.float64(0.8618), 'grid_edge': ['max_depth']}
  XGBoost: R²=0.8631  MAPE=17.74%  MAE=$14.34



Tuning HistGBR...


  [HYPERTUNE  ] attempt 1  {'model': 'HistGradientBoostingRegressor', 'best': {'max_depth': 8}, 'cv_r2': np.float64(0.8852), 'grid_edge': ['max_depth']}


  [HYPERTUNE  ] attempt 2  {'model': 'HistGradientBoostingRegressor', 'best': {'max_depth': 12}, 'cv_r2': np.float64(0.8862), 'grid_edge': []}
  HistGBR: R²=0.8994  MAPE=15.21%  MAE=$12.30



Primary:   HistGradientBoostingRegressor  R²=0.8994
Secondary: XGBRegressor  R²=0.8631


### Split-Conformal Calibration (Shared Calibration Set)

**Process:** Predict on the shared 1776-row calibration set with each model. Compute signed residuals and take 2.5th / 97.5th percentiles as conformal quantiles. This is the FIRST and ONLY time X_cal is used.

**Analysis:** Primary q_low=-40.0039, q_high=44.7300. Secondary q_low=-42.6118, q_high=48.0478. Asymmetric quantiles are expected for skewed revenue distributions. At inference: ci_low = pred + q_low, ci_high = pred + q_high.

**Outcome:** Both models have calibrated conformal quantiles for deployment.

In [10]:
# ------------------------------
# SPLIT-CONFORMAL CALIBRATION (SHARED CALIBRATION SET)
# ------------------------------
# X_cal is touched here for the FIRST and ONLY time in this pipeline.
# Both models calibrated against the SAME 1,776-row held-out set.
alpha = 0.05

def conformal_pair(model):
    y_cal_pred = model.predict(Xcal_t)
    residuals  = np.array(y_cal) - y_cal_pred
    return float(np.quantile(residuals, alpha / 2)), float(np.quantile(residuals, 1 - alpha / 2))

primary_q_low,   primary_q_high   = conformal_pair(primary_entry["model"])
secondary_q_low, secondary_q_high = conformal_pair(secondary_entry["model"])
print(f"Calibration set n: {len(X_cal)}")
print(f"Primary   q_low={primary_q_low:.4f}  q_high={primary_q_high:.4f}")
print(f"Secondary q_low={secondary_q_low:.4f}  q_high={secondary_q_high:.4f}")

# Sanity: empirical coverage on test set
for label, entry, ql, qh in [
    ("primary",   primary_entry,   primary_q_low,   primary_q_high),
    ("secondary", secondary_entry, secondary_q_low, secondary_q_high),
]:
    y_te = np.array(y_test)
    preds = entry["model"].predict(Xte_t)
    covered = float(((y_te >= preds + ql) & (y_te <= preds + qh)).mean())
    print(f"{label} test coverage: {covered:.3f}")

Calibration set n: 1776
Primary   q_low=-40.0039  q_high=44.7300
Secondary q_low=-42.6118  q_high=48.0478
primary test coverage: 0.952
secondary test coverage: 0.943


### Feature Metadata

**Process:** Define the locked `FEATURE_METADATA` list — 12 entries with `name`, `display_name`, and `description`. This dict is the source of truth for human-readable labels in `/info` and the frontend.

**Outcome:** `FEATURE_METADATA` ready for inclusion in the bundle.

In [11]:
# ------------------------------
# FEATURE METADATA — LOCKED TABLE
# ------------------------------
# Source of truth for display labels in /info and the frontend.
# Do NOT auto-generate from data — use this exact dict.
FEATURE_METADATA = [
    {"name": "Department",      "display_name": "Department",    "description": "The retail category the item belongs to"},
    {"name": "Brand_Type",      "display_name": "Brand Type",    "description": "National brand, private label, or specialty"},
    {"name": "Net_Weight_oz",   "display_name": "Net Weight",    "description": "Weight of the item in ounces"},
    {"name": "Pack_Size",       "display_name": "Pack Size",     "description": "Number of units per package"},
    {"name": "List_Price_USD",  "display_name": "List Price",    "description": "Shelf price in dollars before any promotional discount"},
    {"name": "Promo_Price_USD", "display_name": "Promo Price",   "description": "Discounted price during the week — equal to List Price when no promo running"},
    {"name": "Shelf_Facings",   "display_name": "Shelf Facings", "description": "Number of side-by-side product faces on the shelf"},
    {"name": "Store_Number",    "display_name": "Store",         "description": "Which Hill Country Grocer store (HCG-101 through HCG-106)"},
    {"name": "Store_Banner",    "display_name": "Store Banner",  "description": "Format of the store — full grocer, express, or warehouse"},
    {"name": "Store_Region",    "display_name": "Region",        "description": "Texas region — Central, Gulf, or Hill Country"},
    {"name": "Store_Sq_Ft",     "display_name": "Store Size",    "description": "Square footage of the store"},
    {"name": "Store_Open_Year", "display_name": "Store Opened",  "description": "Year the store opened"},
]
print(f"Feature metadata: {len(FEATURE_METADATA)} entries")

Feature metadata: 12 entries


### Save Dual Model Bundle

**Process:** Write `model_bundle/{primary,secondary}/` subdirectories, each containing `model.joblib`, `conformal_q_low.json`, `conformal_q_high.json`, and `metrics.json`. Write shared `calibration_set.parquet` and `feature_metadata.json` at the bundle root.

**Outcome:** Bundle written and SHA-256 confirmed. Structure printed.

In [12]:
# ------------------------------
# SAVE DUAL MODEL BUNDLE
# ------------------------------
# model_bundle/{primary,secondary}/ + shared calibration_set.parquet + feature_metadata.json
import hashlib

_base  = pathlib.Path("/content") if os.path.exists("/content") else pathlib.Path("/tmp")
bundle_root   = _base / "model_bundle"
primary_dir   = bundle_root / "primary"
secondary_dir = bundle_root / "secondary"
for d in [primary_dir, secondary_dir]: d.mkdir(parents=True, exist_ok=True)

sklearn_version = __import__("sklearn").__version__

def write_model_subdir(entry, subdir, ql, qh):
    joblib.dump(
        {"model": entry["model"], "ohe": ohe, "cats": cats, "nums": nums, "feature_names": feature_names},
        subdir / "model.joblib",
    )
    json.dump(ql,  open(subdir / "conformal_q_low.json",  "w"))
    json.dump(qh,  open(subdir / "conformal_q_high.json", "w"))
    json.dump({
        "test_r2":                  entry["test_r2"],
        "test_mape_pct":            entry["test_mape_pct"],
        "test_mae":                 entry["test_mae"],
        "model_family":             entry["model_family"],
        "hyperparams":              entry["hyperparams"],
        "top5_feature_importances": entry["top5_feature_importances"],
    }, open(subdir / "metrics.json", "w"), indent=2)
    return hashlib.sha256(open(subdir / "model.joblib", "rb").read()).hexdigest()

primary_sha   = write_model_subdir(primary_entry,   primary_dir,   primary_q_low,   primary_q_high)
secondary_sha = write_model_subdir(secondary_entry, secondary_dir, secondary_q_low, secondary_q_high)
print(f"primary   model.joblib SHA-256: {primary_sha}")
print(f"secondary model.joblib SHA-256: {secondary_sha}")

X_cal.to_parquet(bundle_root / "calibration_set.parquet", index=False)
print(f"calibration_set.parquet: {len(X_cal)} rows")

json.dump(FEATURE_METADATA, open(bundle_root / "feature_metadata.json", "w"), indent=2)

sample_df = X_train.sample(min(1000, len(X_train)), random_state=SEED)
sample_df.to_parquet(bundle_root / "training_sample.parquet", index=False)
print(f"training_sample.parquet: {len(sample_df)} rows")

print("Bundle tree:")
for p in sorted(bundle_root.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(bundle_root)):45s}  {p.stat().st_size:>10,} bytes")

primary   model.joblib SHA-256: 45b0bbc51a2a1a443cfd19e72f57234a7e0588054212407ff9337f3d091c90fb
secondary model.joblib SHA-256: 473767ddc0b06ebae1a74a2d81a716de0281b8d2a138388db0f7c5504ed54deb


calibration_set.parquet: 1776 rows
training_sample.parquet: 1000 rows
Bundle tree:
  calibration_set.parquet                            26,724 bytes
  feature_metadata.json                               1,655 bytes
  primary/conformal_q_high.json                          17 bytes
  primary/conformal_q_low.json                           19 bytes
  primary/metrics.json                                  634 bytes
  primary/model.joblib                              732,260 bytes
  secondary/conformal_q_high.json                        18 bytes
  secondary/conformal_q_low.json                         18 bytes
  secondary/metrics.json                                618 bytes
  secondary/model.joblib                          2,444,347 bytes
  training_sample.parquet                            19,843 bytes


### Write HF Space Files

**Process:** Write backend (FastAPI `app.py`, `Dockerfile`, `requirements.txt`) and frontend (Streamlit `streamlit_app.py`, `Dockerfile`, `requirements.txt`) files. The frontend includes an Administrator section above a visual divider, with Single/Multi modes below.

**Outcome:** All Space files written and ready for upload.

In [13]:
# ------------------------------
# WRITE HF SPACE FILES
# ------------------------------
# app.py and streamlit_app.py are base64-encoded to avoid triple-quote delimiter issues.
import base64, os, pathlib, shutil

_base        = pathlib.Path("/content") if os.path.exists("/content") else pathlib.Path("/tmp")
_BUILD_STATE = pathlib.Path("/Users/eriksvagshenian/Documents/Claude Cowork/Tasks/hill-country-revenue-forecast-build/build_state")

backend_dir  = _base / "hf-backend"
frontend_dir = _base / "hf-frontend"
backend_dir.mkdir(exist_ok=True)
(frontend_dir / "src").mkdir(parents=True, exist_ok=True)

# Dockerfiles and requirements
(backend_dir / "Dockerfile").write_text(
    "FROM python:3.12-slim\nWORKDIR /app\nCOPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY app.py .\nCOPY model_bundle/ model_bundle/\nCOPY training_sample.parquet .\n"
    "EXPOSE 7860\nCMD [\"uvicorn\", \"app:app\", \"--host\", \"0.0.0.0\", \"--port\", \"7860\"]\n"
)
(backend_dir / "requirements.txt").write_text(
    "fastapi==0.115.5\nuvicorn[standard]==0.32.0\njoblib==1.5.3\n"
    "scikit-learn==1.6.1\nxgboost>=1.7\npandas==2.2.3\nnumpy==2.0.2\npyarrow==17.0.0\n"
)
(frontend_dir / "Dockerfile").write_text(
    "FROM python:3.12-slim\nWORKDIR /app\nCOPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY src/ src/\nEXPOSE 7860\n"
    "CMD [\"streamlit\", \"run\", \"src/streamlit_app.py\", \"--server.port\", \"7860\", \"--server.address\", \"0.0.0.0\"]\n"
)
(frontend_dir / "requirements.txt").write_text(
    "streamlit==1.40.1\nrequests==2.32.3\npandas==2.2.3\npyarrow==17.0.0\n"
)

# app.py and streamlit_app.py: copy from build_state (local) or decode from base64 (Colab)
if (_BUILD_STATE / "backend" / "app.py").exists():
    shutil.copy(_BUILD_STATE / "backend" / "app.py",
                backend_dir / "app.py")
    shutil.copy(_BUILD_STATE / "frontend" / "src" / "streamlit_app.py",
                frontend_dir / "src" / "streamlit_app.py")
    print("Space files: copied from build_state")
else:
    _APP_B64   = "IiIiSGlsbCBDb3VudHJ5IEdyb2NlciDigJQgV2Vla2x5IFJldmVudWUgRm9yZWNhc3QgQVBJICh2My4zLCBkdWFsLW1vZGVsLCBzcGxpdC1jb25mb3JtYWwgQ0kpLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgcGF0aGxpYgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBPcHRpb25hbAoKaW1wb3J0IGpvYmxpYgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgcHlhcnJvdy5wYXJxdWV0IGFzIHBxCmltcG9ydCBza2xlYXJuCmZyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSSwgSFRUUEV4Y2VwdGlvbgpmcm9tIHB5ZGFudGljIGltcG9ydCBCYXNlTW9kZWwKCmFwcCA9IEZhc3RBUEkodGl0bGU9IkhpbGwgQ291bnRyeSBSZXZlbnVlIEZvcmVjYXN0IEFQSSIsIHZlcnNpb249IjMuMy4wIikKCkJVTkRMRV9ST09UID0gcGF0aGxpYi5QYXRoKCJtb2RlbF9idW5kbGUiKQoKCmRlZiBfbG9hZF9tb2RlbF9idW5kbGUoc3ViZGlyOiBzdHIpIC0+IGRpY3Q6CiAgICBkID0gQlVORExFX1JPT1QgLyBzdWJkaXIKICAgIGJ1bmRsZSA9IGpvYmxpYi5sb2FkKGQgLyAibW9kZWwuam9ibGliIikKICAgIHFfbG93ICA9IGpzb24ubG9hZHMoKGQgLyAiY29uZm9ybWFsX3FfbG93Lmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHFfaGlnaCA9IGpzb24ubG9hZHMoKGQgLyAiY29uZm9ybWFsX3FfaGlnaC5qc29uIikucmVhZF90ZXh0KCkpCiAgICBtZXRyaWNzID0ganNvbi5sb2FkcygoZCAvICJtZXRyaWNzLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHJldHVybiB7CiAgICAgICAgIm1vZGVsIjogICBidW5kbGVbIm1vZGVsIl0sCiAgICAgICAgIm9oZSI6ICAgICBidW5kbGVbIm9oZSJdLAogICAgICAgICJjYXRzIjogICAgYnVuZGxlWyJjYXRzIl0sCiAgICAgICAgIm51bXMiOiAgICBidW5kbGVbIm51bXMiXSwKICAgICAgICAicV9sb3ciOiAgIGZsb2F0KHFfbG93KSwKICAgICAgICAicV9oaWdoIjogIGZsb2F0KHFfaGlnaCksCiAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzLAogICAgfQoKCk1PREVMUzogZGljdFtzdHIsIGRpY3RdID0gewogICAgInByaW1hcnkiOiAgIF9sb2FkX21vZGVsX2J1bmRsZSgicHJpbWFyeSIpLAogICAgInNlY29uZGFyeSI6IF9sb2FkX21vZGVsX2J1bmRsZSgic2Vjb25kYXJ5IiksCn0KCl9GRUFUVVJFX01FVEFEQVRBOiBsaXN0W2RpY3RdID0ganNvbi5sb2FkcygKICAgIChCVU5ETEVfUk9PVCAvICJmZWF0dXJlX21ldGFkYXRhLmpzb24iKS5yZWFkX3RleHQoKQopCl9GRUFUVVJFX01FVEFfTUFQOiBkaWN0W3N0ciwgZGljdF0gPSB7ZlsibmFtZSJdOiBmIGZvciBmIGluIF9GRUFUVVJFX01FVEFEQVRBfQoKX1NBTVBMRV9ERjogcGQuRGF0YUZyYW1lID0gcHEucmVhZF90YWJsZSgKICAgIEJVTkRMRV9ST09UIC8gInRyYWluaW5nX3NhbXBsZS5wYXJxdWV0IgopLnRvX3BhbmRhcygpCgpfQ0FMX0RGX0xFTjogaW50ID0gbGVuKAogICAgcHEucmVhZF90YWJsZShCVU5ETEVfUk9PVCAvICJjYWxpYnJhdGlvbl9zZXQucGFycXVldCIpCikKClNLTEVBUk5fVkVSU0lPTjogc3RyID0gc2tsZWFybi5fX3ZlcnNpb25fXwoKCmRlZiBfbWFrZV9kZihyYXc6IGRpY3QsIG1vZGVsX2lkOiBzdHIpIC0+IHBkLkRhdGFGcmFtZToKICAgIG0gPSBNT0RFTFNbbW9kZWxfaWRdCiAgICByb3cgPSB7CiAgICAgICAgY29sOiByYXcuZ2V0KGNvbCwgIlVua25vd24iIGlmIGNvbCBpbiBtWyJjYXRzIl0gZWxzZSAwLjApCiAgICAgICAgZm9yIGNvbCBpbiBtWyJjYXRzIl0gKyBtWyJudW1zIl0KICAgIH0KICAgIHJldHVybiBwZC5EYXRhRnJhbWUoW3Jvd10pW21bImNhdHMiXSArIG1bIm51bXMiXV0KCgpkZWYgX21ha2VfYmF0Y2hfZGYocmVjb3JkczogbGlzdFtkaWN0XSwgbW9kZWxfaWQ6IHN0cikgLT4gcGQuRGF0YUZyYW1lOgogICAgbSA9IE1PREVMU1ttb2RlbF9pZF0KICAgIHJvd3MgPSBbCiAgICAgICAge2NvbDogcmVjLmdldChjb2wsICJVbmtub3duIiBpZiBjb2wgaW4gbVsiY2F0cyJdIGVsc2UgMC4wKQogICAgICAgICBmb3IgY29sIGluIG1bImNhdHMiXSArIG1bIm51bXMiXX0KICAgICAgICBmb3IgcmVjIGluIHJlY29yZHMKICAgIF0KICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cylbbVsiY2F0cyJdICsgbVsibnVtcyJdXQoKCmNsYXNzIFByZWRpY3RSZXF1ZXN0KEJhc2VNb2RlbCk6CiAgICBmZWF0dXJlczogZGljdFtzdHIsIEFueV0KICAgIG1vZGVsOiBPcHRpb25hbFtzdHJdID0gInByaW1hcnkiCgoKY2xhc3MgQmF0Y2hSZXF1ZXN0KEJhc2VNb2RlbCk6CiAgICByZWNvcmRzOiBsaXN0W2RpY3Rbc3RyLCBBbnldXQogICAgbW9kZWw6IE9wdGlvbmFsW3N0cl0gPSAicHJpbWFyeSIKCgpAYXBwLmdldCgiL2hlYWx0aCIpCmRlZiBoZWFsdGgoKToKICAgIHJldHVybiB7InN0YXR1cyI6ICJvayJ9CgoKQGFwcC5nZXQoIi9pbmZvIikKZGVmIGluZm8oKToKICAgIG0gPSBNT0RFTFNbInByaW1hcnkiXQogICAgb2hlX3QgPSBtWyJvaGUiXS5uYW1lZF90cmFuc2Zvcm1lcnNfWyJjIl0KICAgIGNhdHMgID0gbVsiY2F0cyJdCiAgICBudW1zICA9IG1bIm51bXMiXQoKICAgIGZlYXR1cmVzOiBsaXN0W2RpY3RdID0gW10KICAgIGZvciBpZHgsIGNvbCBpbiBlbnVtZXJhdGUoY2F0cyk6CiAgICAgICAgbWV0YSA9IF9GRUFUVVJFX01FVEFfTUFQLmdldChjb2wsIHt9KQogICAgICAgIGZlYXR1cmVzLmFwcGVuZCh7CiAgICAgICAgICAgICJuYW1lIjogICAgICAgICAgIGNvbCwKICAgICAgICAgICAgImRpc3BsYXlfbmFtZSI6ICAgbWV0YS5nZXQoImRpc3BsYXlfbmFtZSIsIGNvbCksCiAgICAgICAgICAgICJkZXNjcmlwdGlvbiI6ICAgIG1ldGEuZ2V0KCJkZXNjcmlwdGlvbiIsICIiKSwKICAgICAgICAgICAgInR5cGUiOiAgICAgICAgICAgImNhdGVnb3JpY2FsIiwKICAgICAgICAgICAgImFsbG93ZWRfdmFsdWVzIjogW3N0cih2KSBmb3IgdiBpbiBvaGVfdC5jYXRlZ29yaWVzX1tpZHhdXSwKICAgICAgICB9KQogICAgZm9yIGNvbCBpbiBudW1zOgogICAgICAgIG1ldGEgPSBfRkVBVFVSRV9NRVRBX01BUC5nZXQoY29sLCB7fSkKICAgICAgICBjb2xfZGF0YSA9IF9TQU1QTEVfREZbY29sXSBpZiBjb2wgaW4gX1NBTVBMRV9ERi5jb2x1bW5zIGVsc2UgTm9uZQogICAgICAgIGVudHJ5OiBkaWN0ID0gewogICAgICAgICAgICAibmFtZSI6ICAgICAgICAgY29sLAogICAgICAgICAgICAiZGlzcGxheV9uYW1lIjogbWV0YS5nZXQoImRpc3BsYXlfbmFtZSIsIGNvbCksCiAgICAgICAgICAgICJkZXNjcmlwdGlvbiI6ICBtZXRhLmdldCgiZGVzY3JpcHRpb24iLCAiIiksCiAgICAgICAgICAgICJ0eXBlIjogICAgICAgICAibnVtZXJpYyIsCiAgICAgICAgfQogICAgICAgIGlmIGNvbF9kYXRhIGlzIG5vdCBOb25lOgogICAgICAgICAgICBlbnRyeVsibWluIl0gPSByb3VuZChmbG9hdChjb2xfZGF0YS5taW4oKSksIDQpCiAgICAgICAgICAgIGVudHJ5WyJtYXgiXSA9IHJvdW5kKGZsb2F0KGNvbF9kYXRhLm1heCgpKSwgNCkKICAgICAgICBmZWF0dXJlcy5hcHBlbmQoZW50cnkpCgogICAgbW9kZWxzX2luZm86IGxpc3RbZGljdF0gPSBbXQogICAgZm9yIG1vZGVsX2lkLCBtb2RlbF9kYXRhIGluIE1PREVMUy5pdGVtcygpOgogICAgICAgIG1ldHJpY3MgPSBtb2RlbF9kYXRhWyJtZXRyaWNzIl0KICAgICAgICBtb2RlbHNfaW5mby5hcHBlbmQoewogICAgICAgICAgICAiaWQiOiAgICAgICAgICAgICAgICAgICAgICBtb2RlbF9pZCwKICAgICAgICAgICAgIm1vZGVsX2ZhbWlseSI6ICAgICAgICAgICAgbWV0cmljc1sibW9kZWxfZmFtaWx5Il0sCiAgICAgICAgICAgICJ0ZXN0X3IyIjogICAgICAgICAgICAgICAgIHJvdW5kKGZsb2F0KG1ldHJpY3NbInRlc3RfcjIiXSksIDQpLAogICAgICAgICAgICAidGVzdF9tYXBlX3BjdCI6ICAgICAgICAgICByb3VuZChmbG9hdChtZXRyaWNzWyJ0ZXN0X21hcGVfcGN0Il0pLCAyKSwKICAgICAgICAgICAgInRlc3RfbWFlIjogICAgICAgICAgICAgICAgcm91bmQoZmxvYXQobWV0cmljc1sidGVzdF9tYWUiXSksIDIpLAogICAgICAgICAgICAiY2lfcV9sb3ciOiAgICAgICAgICAgICAgICByb3VuZChmbG9hdChtb2RlbF9kYXRhWyJxX2xvdyJdKSwgNCksCiAgICAgICAgICAgICJjaV9xX2hpZ2giOiAgICAgICAgICAgICAgIHJvdW5kKGZsb2F0KG1vZGVsX2RhdGFbInFfaGlnaCJdKSwgNCksCiAgICAgICAgICAgICJ0b3A1X2ZlYXR1cmVfaW1wb3J0YW5jZXMiOiBtZXRyaWNzLmdldCgidG9wNV9mZWF0dXJlX2ltcG9ydGFuY2VzIiwgW10pLAogICAgICAgIH0pCgogICAgcmV0dXJuIHsKICAgICAgICAibW9kZWxzIjogICAgICAgICAgIG1vZGVsc19pbmZvLAogICAgICAgICJ0YXJnZXQiOiAgICAgICAgICAgIldlZWtseV9SZXZlbnVlX1VTRCIsCiAgICAgICAgInNlY29uZGFyeV90YXJnZXQiOiAiV2Vla2x5X1VuaXRzX1NvbGQiLAogICAgICAgICJza2xlYXJuX3ZlcnNpb24iOiAgU0tMRUFSTl9WRVJTSU9OLAogICAgICAgICJjaV9tZWNoYW5pc20iOiAgICAgInNwbGl0X2NvbmZvcm1hbCIsCiAgICAgICAgImNpX2FscGhhIjogICAgICAgICAwLjA1LAogICAgICAgICJjaV9jb3ZlcmFnZV9wY3QiOiAgOTUsCiAgICAgICAgImNpX2NhbGlicmF0aW9uX24iOiBfQ0FMX0RGX0xFTiwKICAgICAgICAiZmVhdHVyZXMiOiAgICAgICAgIGZlYXR1cmVzLAogICAgfQoKCkBhcHAucG9zdCgiL3ByZWRpY3QiKQpkZWYgcHJlZGljdChyZXE6IFByZWRpY3RSZXF1ZXN0KToKICAgIG1vZGVsX2lkID0gcmVxLm1vZGVsIGlmIHJlcS5tb2RlbCBpbiBNT0RFTFMgZWxzZSAicHJpbWFyeSIKICAgIG0gPSBNT0RFTFNbbW9kZWxfaWRdCiAgICB0cnk6CiAgICAgICAgZGYgICA9IF9tYWtlX2RmKHJlcS5mZWF0dXJlcywgbW9kZWxfaWQpCiAgICAgICAgcHJlZCA9IGZsb2F0KG1bIm1vZGVsIl0ucHJlZGljdChtWyJvaGUiXS50cmFuc2Zvcm0oZGYpKVswXSkKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAicHJlZGljdGlvbiI6IHJvdW5kKHByZWQsIDIpLAogICAgICAgICAgICAiY2lfbG93IjogICAgIHJvdW5kKHByZWQgKyBtWyJxX2xvdyJdLCAgMiksCiAgICAgICAgICAgICJjaV9oaWdoIjogICAgcm91bmQocHJlZCArIG1bInFfaGlnaCJdLCAyKSwKICAgICAgICAgICAgImN1cnJlbmN5IjogICAiVVNEIiwKICAgICAgICAgICAgIm1vZGVsIjogICAgICBtb2RlbF9pZCwKICAgICAgICB9CiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTQyMiwgZGV0YWlsPXN0cihleGMpKQoKCkBhcHAucG9zdCgiL3ByZWRpY3RfYmF0Y2giKQpkZWYgcHJlZGljdF9iYXRjaChyZXE6IEJhdGNoUmVxdWVzdCk6CiAgICBpZiBub3QgcmVxLnJlY29yZHM6CiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT00MDAsIGRldGFpbD0icmVjb3JkcyBjYW5ub3QgYmUgZW1wdHkiKQogICAgbW9kZWxfaWQgPSByZXEubW9kZWwgaWYgcmVxLm1vZGVsIGluIE1PREVMUyBlbHNlICJwcmltYXJ5IgogICAgbSA9IE1PREVMU1ttb2RlbF9pZF0KICAgIHRyeToKICAgICAgICBkZiAgICA9IF9tYWtlX2JhdGNoX2RmKHJlcS5yZWNvcmRzLCBtb2RlbF9pZCkKICAgICAgICBwcmVkcyA9IG1bIm1vZGVsIl0ucHJlZGljdChtWyJvaGUiXS50cmFuc2Zvcm0oZGYpKQogICAgICAgIHJlc3VsdHMgPSBbCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJwcmVkaWN0aW9uIjogcm91bmQoZmxvYXQocCksIDIpLAogICAgICAgICAgICAgICAgImNpX2xvdyI6ICAgICByb3VuZChmbG9hdChwKSArIG1bInFfbG93Il0sICAyKSwKICAgICAgICAgICAgICAgICJjaV9oaWdoIjogICAgcm91bmQoZmxvYXQocCkgKyBtWyJxX2hpZ2giXSwgMiksCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIHAgaW4gcHJlZHMKICAgICAgICBdCiAgICAgICAgcmV0dXJuIHsicHJlZGljdGlvbnMiOiByZXN1bHRzLCAiY291bnQiOiBsZW4ocmVzdWx0cyksICJtb2RlbCI6IG1vZGVsX2lkfQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT00MjIsIGRldGFpbD1zdHIoZXhjKSkKCgpAYXBwLmdldCgiL3RyYWluaW5nX3NhbXBsZSIpCmRlZiB0cmFpbmluZ19zYW1wbGUoKToKICAgIHJldHVybiB7InNhbXBsZSI6IF9TQU1QTEVfREYudG9fZGljdChvcmllbnQ9InJlY29yZHMiKSwgIm4iOiBsZW4oX1NBTVBMRV9ERil9Cg=="
    _STLIT_B64 = "IiIiSGlsbCBDb3VudHJ5IEdyb2NlciBSZXZlbnVlIEZvcmVjYXN0IOKAlCBTdHJlYW1saXQgZnJvbnRlbmQgKHYzLjMpLiIiIgppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCByZXF1ZXN0cwppbXBvcnQgc3RyZWFtbGl0IGFzIHN0CgpCQUNLRU5EX1VSTCA9ICJodHRwczovL2V2YWdhaW1sLWhpbGwtY291bnRyeS1yZXZlbnVlLWZvcmVjYXN0LWJhY2tlbmQuaGYuc3BhY2UiCgpzdC5zZXRfcGFnZV9jb25maWcoCiAgICBwYWdlX3RpdGxlPSJIaWxsIENvdW50cnkgR3JvY2VyIFJldmVudWUgRm9yZWNhc3QiLAogICAgcGFnZV9pY29uPSLwn5uSIiwKICAgIGxheW91dD0id2lkZSIsCikKCgpAc3QuY2FjaGVfZGF0YSh0dGw9MzYwMCkKZGVmIGZldGNoX2luZm8oKSAtPiBkaWN0OgogICAgciA9IHJlcXVlc3RzLmdldChmIntCQUNLRU5EX1VSTH0vaW5mbyIsIHRpbWVvdXQ9MzApCiAgICByLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgcmV0dXJuIHIuanNvbigpCgoKQHN0LmNhY2hlX2RhdGEodHRsPTM2MDApCmRlZiBmZXRjaF9zYW1wbGUoKSAtPiBwZC5EYXRhRnJhbWU6CiAgICByID0gcmVxdWVzdHMuZ2V0KGYie0JBQ0tFTkRfVVJMfS90cmFpbmluZ19zYW1wbGUiLCB0aW1lb3V0PTMwKQogICAgci5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgIHJldHVybiBwZC5EYXRhRnJhbWUoci5qc29uKClbInNhbXBsZSJdKQoKCmRlZiBwcmVkaWN0X29uZShmZWF0dXJlczogZGljdCwgbW9kZWxfaWQ6IHN0cikgLT4gZGljdDoKICAgIHIgPSByZXF1ZXN0cy5wb3N0KAogICAgICAgIGYie0JBQ0tFTkRfVVJMfS9wcmVkaWN0IiwKICAgICAgICBqc29uPXsiZmVhdHVyZXMiOiBmZWF0dXJlcywgIm1vZGVsIjogbW9kZWxfaWR9LAogICAgICAgIHRpbWVvdXQ9MzAsCiAgICApCiAgICByLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgcmV0dXJuIHIuanNvbigpCgoKZGVmIHByZWRpY3RfYmF0Y2hfYXBpKHJlY29yZHM6IGxpc3RbZGljdF0sIG1vZGVsX2lkOiBzdHIpIC0+IGRpY3Q6CiAgICByID0gcmVxdWVzdHMucG9zdCgKICAgICAgICBmIntCQUNLRU5EX1VSTH0vcHJlZGljdF9iYXRjaCIsCiAgICAgICAganNvbj17InJlY29yZHMiOiByZWNvcmRzLCAibW9kZWwiOiBtb2RlbF9pZH0sCiAgICAgICAgdGltZW91dD02MCwKICAgICkKICAgIHIucmFpc2VfZm9yX3N0YXR1cygpCiAgICByZXR1cm4gci5qc29uKCkKCgp0cnk6CiAgICBpbmZvID0gZmV0Y2hfaW5mbygpCiAgICBzYW1wbGVfZGYgPSBmZXRjaF9zYW1wbGUoKQpleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgIHN0LmVycm9yKGYiQmFja2VuZCB1bmF2YWlsYWJsZToge2V4Y30iKQogICAgc3Quc3RvcCgpCgpmbWFwOiBkaWN0W3N0ciwgZGljdF0gPSB7ZlsibmFtZSJdOiBmIGZvciBmIGluIGluZm9bImZlYXR1cmVzIl19Cm1vZGVsc19saXN0OiBsaXN0W2RpY3RdID0gaW5mby5nZXQoIm1vZGVscyIsIFtdKQoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBBRE1JTklTVFJBVE9SIFNFQ1RJT04gIChhYm92ZSBkaXZpZGVyKQojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApzdC5tYXJrZG93bigiIyMgQURNSU5JU1RSQVRPUiIpCgptb2RlbF9sYWJlbHMgPSBbCiAgICBmInttWydtb2RlbF9mYW1pbHknXX0gKFLCsj17bVsndGVzdF9yMiddOi4yZn0pIiBmb3IgbSBpbiBtb2RlbHNfbGlzdApdCm1vZGVsX2lkcyA9IFttWyJpZCJdIGZvciBtIGluIG1vZGVsc19saXN0XQoKc2VsX2xhYmVsID0gc3Quc2VsZWN0Ym94KCJNb2RlbCIsIG1vZGVsX2xhYmVscywga2V5PSJtb2RlbF9zZWxlY3RvciIpCnNlbGVjdGVkX2lkeCA9IG1vZGVsX2xhYmVscy5pbmRleChzZWxfbGFiZWwpIGlmIHNlbF9sYWJlbCBpbiBtb2RlbF9sYWJlbHMgZWxzZSAwCnNlbGVjdGVkX21vZGVsX2lkID0gbW9kZWxfaWRzW3NlbGVjdGVkX2lkeF0Kc3Quc2Vzc2lvbl9zdGF0ZVsic2VsZWN0ZWRfbW9kZWxfaWQiXSA9IHNlbGVjdGVkX21vZGVsX2lkCgpzdC5tYXJrZG93bigiKipCYXRjaCB1cGxvYWQqKiIpCnJlcV9jb2xzID0gW2ZbIm5hbWUiXSBmb3IgZiBpbiBpbmZvWyJmZWF0dXJlcyJdXQp1cGxvYWRlZCA9IHN0LmZpbGVfdXBsb2FkZXIoIkNob29zZSBDU1YgZmlsZSIsIHR5cGU9WyJjc3YiXSwga2V5PSJhZG1pbl9iYXRjaF91cGxvYWQiKQppZiB1cGxvYWRlZDoKICAgIGRmX3VwID0gcGQucmVhZF9jc3YodXBsb2FkZWQpCiAgICBtaXNzaW5nID0gW2MgZm9yIGMgaW4gcmVxX2NvbHMgaWYgYyBub3QgaW4gZGZfdXAuY29sdW1uc10KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgc3Qud2FybmluZyhmIk1pc3NpbmcgcmVxdWlyZWQgY29sdW1uczoge21pc3Npbmd9IikKICAgIGVsc2U6CiAgICAgICAgIyBTY2hlbWEgdmFsaWRhdGlvbgogICAgICAgIHZhbF9lcnJvcnMgPSBbXQogICAgICAgIGZvciBmZWF0IGluIGluZm9bImZlYXR1cmVzIl06CiAgICAgICAgICAgIGNvbCA9IGZlYXRbIm5hbWUiXQogICAgICAgICAgICBpZiBmZWF0WyJ0eXBlIl0gPT0gImNhdGVnb3JpY2FsIiBhbmQgY29sIGluIGRmX3VwLmNvbHVtbnM6CiAgICAgICAgICAgICAgICBhbGxvd2VkID0gc2V0KGZlYXRbImFsbG93ZWRfdmFsdWVzIl0pCiAgICAgICAgICAgICAgICBiYWQgPSBbdiBmb3IgdiBpbiBkZl91cFtjb2xdLmFzdHlwZShzdHIpLnVuaXF1ZSgpIGlmIHYgbm90IGluIGFsbG93ZWRdCiAgICAgICAgICAgICAgICBpZiBiYWQ6CiAgICAgICAgICAgICAgICAgICAgdmFsX2Vycm9ycy5hcHBlbmQoZiJ7Y29sfTogdW5rbm93biB2YWx1ZXMge2JhZFs6M119IikKICAgICAgICAgICAgZWxpZiBmZWF0WyJ0eXBlIl0gPT0gIm51bWVyaWMiIGFuZCBjb2wgaW4gZGZfdXAuY29sdW1uczoKICAgICAgICAgICAgICAgIGxvID0gZmVhdC5nZXQoIm1pbiIsIGZsb2F0KCItaW5mIikpCiAgICAgICAgICAgICAgICBoaSA9IGZlYXQuZ2V0KCJtYXgiLCBmbG9hdCgiaW5mIikpCiAgICAgICAgICAgICAgICBmYXJfb3V0ID0gaW50KCgoZGZfdXBbY29sXSA8IGxvICogMC41KSB8IChkZl91cFtjb2xdID4gaGkgKiAyKSkuc3VtKCkpCiAgICAgICAgICAgICAgICBpZiBmYXJfb3V0OgogICAgICAgICAgICAgICAgICAgIHZhbF9lcnJvcnMuYXBwZW5kKGYie2NvbH06IHtmYXJfb3V0fSByb3dzIGZhciBvdXRzaWRlIHRyYWluaW5nIHJhbmdlIikKICAgICAgICBmb3IgZXJyIGluIHZhbF9lcnJvcnM6CiAgICAgICAgICAgIHN0Lndhcm5pbmcoZXJyKQogICAgICAgIHN0LmRhdGFmcmFtZShkZl91cC5oZWFkKDUpLCBoaWRlX2luZGV4PVRydWUpCiAgICAgICAgaWYgc3QuYnV0dG9uKCJQcmVkaWN0IGJhdGNoIiwgdHlwZT0icHJpbWFyeSIsIGtleT0iYWRtaW5fYmF0Y2hfYnRuIik6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHJlcyA9IHByZWRpY3RfYmF0Y2hfYXBpKAogICAgICAgICAgICAgICAgICAgIGRmX3VwW3JlcV9jb2xzXS50b19kaWN0KG9yaWVudD0icmVjb3JkcyIpLAogICAgICAgICAgICAgICAgICAgIHNlbGVjdGVkX21vZGVsX2lkLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZGZfb3V0ID0gZGZfdXAuY29weSgpCiAgICAgICAgICAgICAgICBkZl9vdXRbInByZWRpY3Rpb25fdXNkIl0gPSBbcFsicHJlZGljdGlvbiJdIGZvciBwIGluIHJlc1sicHJlZGljdGlvbnMiXV0KICAgICAgICAgICAgICAgIGRmX291dFsiY2lfbG93X3VzZCJdICAgICA9IFtwWyJjaV9sb3ciXSAgICAgZm9yIHAgaW4gcmVzWyJwcmVkaWN0aW9ucyJdXQogICAgICAgICAgICAgICAgZGZfb3V0WyJjaV9oaWdoX3VzZCJdICAgID0gW3BbImNpX2hpZ2giXSAgICBmb3IgcCBpbiByZXNbInByZWRpY3Rpb25zIl1dCiAgICAgICAgICAgICAgICBzdC5zdWNjZXNzKAogICAgICAgICAgICAgICAgICAgIGYie3Jlc1snY291bnQnXX0gcHJlZGljdGlvbnMgwrcgbW9kZWw6IHtyZXNbJ21vZGVsJ119IMK3ICIKICAgICAgICAgICAgICAgICAgICBmInRvdGFsOiAke2RmX291dFsncHJlZGljdGlvbl91c2QnXS5zdW0oKTosLjJmfSIKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIHN0LmRhdGFmcmFtZShkZl9vdXQsIGhpZGVfaW5kZXg9VHJ1ZSkKICAgICAgICAgICAgICAgIHN0LmRvd25sb2FkX2J1dHRvbigKICAgICAgICAgICAgICAgICAgICAiRG93bmxvYWQgZW5yaWNoZWQgQ1NWIiwKICAgICAgICAgICAgICAgICAgICBkZl9vdXQudG9fY3N2KGluZGV4PUZhbHNlKS5lbmNvZGUoKSwKICAgICAgICAgICAgICAgICAgICAiaGlsbF9jb3VudHJ5X3ByZWRpY3Rpb25zLmNzdiIsCiAgICAgICAgICAgICAgICAgICAgInRleHQvY3N2IiwKICAgICAgICAgICAgICAgICAgICBrZXk9ImFkbWluX2Rvd25sb2FkIiwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICBzdC5lcnJvcihzdHIoZXhjKSkKCnN0LmRpdmlkZXIoKQoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBNRVJDSEFORElTRVIgU1VSRkFDRSAgKGJlbG93IGRpdmlkZXIpCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACnN0LnRpdGxlKCJIaWxsIENvdW50cnkgR3JvY2VyIOKAlCBXZWVrbHkgUmV2ZW51ZSBGb3JlY2FzdCIpCnN0LmNhcHRpb24oIlNwbGl0LWNvbmZvcm1hbCA5NSUgcHJlZGljdGlvbiBpbnRlcnZhbHMgwrcgUG93ZXJlZCBieSBFdmFnQUlNTCIpCgptb2RlID0gc3Quc2VnbWVudGVkX2NvbnRyb2woIk1vZGUiLCBbIlNpbmdsZSIsICJNdWx0aSJdLCBkZWZhdWx0PSJTaW5nbGUiKQoKCmRlZiBfcmVuZGVyX2ZlYXR1cmUoZmVhdDogZGljdCwgcHJlZml4OiBzdHIsIGRlZmF1bHQ9Tm9uZSk6CiAgICAiIiJCb2xkIGRpc3BsYXlfbmFtZSBoZWFkZXIgKyBzbWFsbCBjb2x1bW4g4oCUIGRlc2NyaXB0aW9uLCB0aGVuIHRoZSBpbnB1dCB3aWRnZXQuIiIiCiAgICBzdC5tYXJrZG93bihmIioqe2ZlYXRbJ2Rpc3BsYXlfbmFtZSddfSoqIikKICAgIHN0Lm1hcmtkb3duKAogICAgICAgIGYiPHNwYW4gc3R5bGU9J2ZvbnQtc2l6ZTowLjg1ZW07Y29sb3I6Izg4ODsnPiIKICAgICAgICBmIntmZWF0WyduYW1lJ119IOKAlCB7ZmVhdFsnZGVzY3JpcHRpb24nXX08L3NwYW4+IiwKICAgICAgICB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlLAogICAgKQogICAga2V5ID0gZiJ7cHJlZml4fV9fe2ZlYXRbJ25hbWUnXX0iCiAgICBpZiBmZWF0WyJ0eXBlIl0gPT0gImNhdGVnb3JpY2FsIjoKICAgICAgICBvcHRzID0gZmVhdFsiYWxsb3dlZF92YWx1ZXMiXQogICAgICAgIGlkeCA9IG9wdHMuaW5kZXgoZGVmYXVsdCkgaWYgZGVmYXVsdCBpbiBvcHRzIGVsc2UgMAogICAgICAgIHJldHVybiBzdC5zZWxlY3Rib3goCiAgICAgICAgICAgICIiLCBvcHRzLCBpbmRleD1pZHgsIGtleT1rZXksIGxhYmVsX3Zpc2liaWxpdHk9ImNvbGxhcHNlZCIKICAgICAgICApCiAgICBlbHNlOgogICAgICAgIGxvICA9IGZsb2F0KGZlYXQuZ2V0KCJtaW4iLCAwLjApKQogICAgICAgIGhpICA9IGZsb2F0KGZlYXQuZ2V0KCJtYXgiLCAxMDAwLjApKQogICAgICAgIHZhbCA9IGZsb2F0KGRlZmF1bHQpIGlmIGRlZmF1bHQgaXMgbm90IE5vbmUgZWxzZSBsbwogICAgICAgIHN0ZXAgPSAwLjEgaWYgaXNpbnN0YW5jZShsbywgZmxvYXQpIG9yIGlzaW5zdGFuY2UoaGksIGZsb2F0KSBlbHNlIDEuMAogICAgICAgIHJldHVybiBzdC5udW1iZXJfaW5wdXQoCiAgICAgICAgICAgICIiLCBtaW5fdmFsdWU9bG8gKiAwLjUsIG1heF92YWx1ZT1oaSAqIDIuMCwKICAgICAgICAgICAgdmFsdWU9dmFsLCBzdGVwPXN0ZXAsIGtleT1rZXksIGxhYmVsX3Zpc2liaWxpdHk9ImNvbGxhcHNlZCIsCiAgICAgICAgKQoKCmRlZiBzdG9yZV9maWVsZHMocHJlZml4OiBzdHIsIGRmX3NhbXBsZTogcGQuRGF0YUZyYW1lKSAtPiBkaWN0OgogICAgc3JjID0gc3QucmFkaW8oCiAgICAgICAgIlN0b3JlIHNvdXJjZSIsIFsiRXhpc3Rpbmcgc3RvcmUiLCAiTmV3IHN0b3JlIl0sCiAgICAgICAga2V5PWYie3ByZWZpeH1fX3NyYyIsIGhvcml6b250YWw9VHJ1ZSwKICAgICkKICAgIHNmOiBkaWN0ID0ge30KICAgIGlmIHNyYyA9PSAiRXhpc3Rpbmcgc3RvcmUiOgogICAgICAgIG9wdHMgPSBmbWFwWyJTdG9yZV9OdW1iZXIiXVsiYWxsb3dlZF92YWx1ZXMiXQogICAgICAgIHNuID0gc3Quc2VsZWN0Ym94KCIqKlN0b3JlKioiLCBvcHRzLCBrZXk9ZiJ7cHJlZml4fV9fc24iKQogICAgICAgIHJvdyA9IGRmX3NhbXBsZVtkZl9zYW1wbGVbIlN0b3JlX051bWJlciJdID09IHNuXQogICAgICAgIHJvdyA9IHJvdy5pbG9jWzBdIGlmIGxlbihyb3cpIGVsc2UgZGZfc2FtcGxlLmlsb2NbMF0KICAgICAgICBzZiA9IHsKICAgICAgICAgICAgIlN0b3JlX051bWJlciI6ICAgIHNuLAogICAgICAgICAgICAiU3RvcmVfQmFubmVyIjogICAgc3RyKHJvdy5nZXQoIlN0b3JlX0Jhbm5lciIsICAgICJIaWxsIENvdW50cnkgR3JvY2VyIikpLAogICAgICAgICAgICAiU3RvcmVfUmVnaW9uIjogICAgc3RyKHJvdy5nZXQoIlN0b3JlX1JlZ2lvbiIsICAgICJTb3V0aCAtIFRleGFzIENlbnRyYWwiKSksCiAgICAgICAgICAgICJTdG9yZV9TcV9GdCI6ICAgICBmbG9hdChyb3cuZ2V0KCJTdG9yZV9TcV9GdCIsICAgICA0MjAwMCkpLAogICAgICAgICAgICAiU3RvcmVfT3Blbl9ZZWFyIjogaW50KHJvdy5nZXQoIlN0b3JlX09wZW5fWWVhciIsIDIwMTEpKSwKICAgICAgICB9CiAgICAgICAgc3QuY2FwdGlvbigKICAgICAgICAgICAgZiJCYW5uZXI6IHtzZlsnU3RvcmVfQmFubmVyJ119IMK3IFJlZ2lvbjoge3NmWydTdG9yZV9SZWdpb24nXX0gwrcgIgogICAgICAgICAgICBmIntpbnQoc2ZbJ1N0b3JlX1NxX0Z0J10pOix9IHNxIGZ0IgogICAgICAgICkKICAgIGVsc2U6CiAgICAgICAgYzEsIGMyID0gc3QuY29sdW1ucygyKQogICAgICAgIHdpdGggYzE6CiAgICAgICAgICAgIHNmWyJTdG9yZV9CYW5uZXIiXSA9IF9yZW5kZXJfZmVhdHVyZShmbWFwWyJTdG9yZV9CYW5uZXIiXSwgZiJ7cHJlZml4fV9iYW5uZXIiKQogICAgICAgIHdpdGggYzI6CiAgICAgICAgICAgIHNmWyJTdG9yZV9SZWdpb24iXSA9IF9yZW5kZXJfZmVhdHVyZShmbWFwWyJTdG9yZV9SZWdpb24iXSwgZiJ7cHJlZml4fV9yZWdpb24iKQogICAgICAgIHNmWyJTdG9yZV9OdW1iZXIiXSA9ICJIQ0ctMTAxIgogICAgICAgIG4xLCBuMiA9IHN0LmNvbHVtbnMoMikKICAgICAgICB3aXRoIG4xOgogICAgICAgICAgICBzZlsiU3RvcmVfU3FfRnQiXSAgICAgPSBfcmVuZGVyX2ZlYXR1cmUoZm1hcFsiU3RvcmVfU3FfRnQiXSwgICAgIGYie3ByZWZpeH1fc3FmdCIpCiAgICAgICAgd2l0aCBuMjoKICAgICAgICAgICAgc2ZbIlN0b3JlX09wZW5fWWVhciJdID0gX3JlbmRlcl9mZWF0dXJlKGZtYXBbIlN0b3JlX09wZW5fWWVhciJdLCBmIntwcmVmaXh9X3lyIikKICAgIHJldHVybiBzZgoKCmRlZiBpdGVtX2ZpZWxkcyhwcmVmaXg6IHN0cikgLT4gZGljdDoKICAgIGl0ZW1fZmVhdF9uYW1lcyA9IFsKICAgICAgICAiRGVwYXJ0bWVudCIsICJCcmFuZF9UeXBlIiwgIk5ldF9XZWlnaHRfb3oiLCAiUGFja19TaXplIiwKICAgICAgICAiTGlzdF9QcmljZV9VU0QiLCAiUHJvbW9fUHJpY2VfVVNEIiwgIlNoZWxmX0ZhY2luZ3MiLAogICAgXQogICAgcmVzdWx0OiBkaWN0ID0ge30KICAgIGNvbHMgPSBzdC5jb2x1bW5zKDIpCiAgICBmb3IgaSwgZm5hbWUgaW4gZW51bWVyYXRlKGl0ZW1fZmVhdF9uYW1lcyk6CiAgICAgICAgd2l0aCBjb2xzW2kgJSAyXToKICAgICAgICAgICAgcmVzdWx0W2ZuYW1lXSA9IF9yZW5kZXJfZmVhdHVyZShmbWFwW2ZuYW1lXSwgcHJlZml4KQogICAgcmV0dXJuIHJlc3VsdAoKCm1pZCA9IHN0LnNlc3Npb25fc3RhdGUuZ2V0KCJzZWxlY3RlZF9tb2RlbF9pZCIsICJwcmltYXJ5IikKCmlmIG1vZGUgPT0gIlNpbmdsZSI6CiAgICBzdG9yZSA9IHN0b3JlX2ZpZWxkcygicyIsIHNhbXBsZV9kZikKICAgIGl0ZW0gID0gaXRlbV9maWVsZHMoInNpIikKICAgIGZlYXR1cmVzID0geyoqc3RvcmUsICoqaXRlbX0KICAgIGlmIHN0LmJ1dHRvbigiUHJlZGljdCIsIHR5cGU9InByaW1hcnkiKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlcyA9IHByZWRpY3Rfb25lKGZlYXR1cmVzLCBtaWQpCiAgICAgICAgICAgIHAsIGxvLCBoaSA9IHJlc1sicHJlZGljdGlvbiJdLCByZXNbImNpX2xvdyJdLCByZXNbImNpX2hpZ2giXQogICAgICAgICAgICBodyA9IChoaSAtIGxvKSAvIDIuMAogICAgICAgICAgICBzdC5zdWNjZXNzKGYiKipQcmVkaWN0ZWQgd2Vla2x5IHJldmVudWU6ICR7cDosLjJmfSDCsSAke2h3OiwuMmZ9KioiKQogICAgICAgICAgICBzdC5jYXB0aW9uKGYiOTUlIFBJOiAke2xvOiwuMmZ9IOKAkyAke2hpOiwuMmZ9IMK3IG1vZGVsOiB7cmVzWydtb2RlbCddfSIpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgIHN0LmVycm9yKHN0cihleGMpKQoKZWxpZiBtb2RlID09ICJNdWx0aSI6CiAgICBuX2l0ZW1zID0gaW50KHN0Lm51bWJlcl9pbnB1dCgiSXRlbXMgaW4gYmFza2V0IiwgMSwgMTAsIDIpKQogICAgc3RvcmUgPSBzdG9yZV9maWVsZHMoIm0iLCBzYW1wbGVfZGYpCiAgICBpdGVtczogbGlzdFtkaWN0XSA9IFtdCiAgICBmb3IgaSBpbiByYW5nZShuX2l0ZW1zKToKICAgICAgICB3aXRoIHN0LmV4cGFuZGVyKGYiSXRlbSB7aSArIDF9IiwgZXhwYW5kZWQ9VHJ1ZSk6CiAgICAgICAgICAgIGl0ZW1zLmFwcGVuZChpdGVtX2ZpZWxkcyhmIm1pe2l9IikpCiAgICByZWNvcmRzID0gW3sqKnN0b3JlLCAqKml0fSBmb3IgaXQgaW4gaXRlbXNdCiAgICBjMSwgYzIgPSBzdC5jb2x1bW5zKDIpCiAgICBpZiBjMS5idXR0b24oIlByZWRpY3QgYmFza2V0IiwgdHlwZT0icHJpbWFyeSIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzID0gcHJlZGljdF9iYXRjaF9hcGkocmVjb3JkcywgbWlkKQogICAgICAgICAgICBwcmVkcyA9IHJlc1sicHJlZGljdGlvbnMiXQogICAgICAgICAgICB0b3RhbCA9IHN1bShwWyJwcmVkaWN0aW9uIl0gZm9yIHAgaW4gcHJlZHMpCiAgICAgICAgICAgIHN0LnN1Y2Nlc3MoCiAgICAgICAgICAgICAgICBmIioqQmFza2V0IHRvdGFsOiAke3RvdGFsOiwuMmZ9L3dlZWsqKiDCtyBtb2RlbDoge3Jlc1snbW9kZWwnXX0iCiAgICAgICAgICAgICkKICAgICAgICAgICAgcm93cyA9IFsKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAiSXRlbSI6ICAgIGkgKyAxLAogICAgICAgICAgICAgICAgICAgICJEZXB0IjogICAgcmVjb3Jkc1tpXVsiRGVwYXJ0bWVudCJdLAogICAgICAgICAgICAgICAgICAgICJSZXZlbnVlIjogZiIke3BbJ3ByZWRpY3Rpb24nXTosLjJmfSIsCiAgICAgICAgICAgICAgICAgICAgIkNJIMKxIjogICAgZiIkeyhwWydjaV9oaWdoJ10gLSBwWydjaV9sb3cnXSkgLyAyOiwuMmZ9IiwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGZvciBpLCBwIGluIGVudW1lcmF0ZShwcmVkcykKICAgICAgICAgICAgXQogICAgICAgICAgICBzdC5kYXRhZnJhbWUocGQuRGF0YUZyYW1lKHJvd3MpLCBoaWRlX2luZGV4PVRydWUpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgIHN0LmVycm9yKHN0cihleGMpKQogICAgaWYgYzIuYnV0dG9uKCJBY3Jvc3MgYWxsIHN0b3JlcyIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgc3RvcmVfbnVtcyA9IGZtYXBbIlN0b3JlX051bWJlciJdWyJhbGxvd2VkX3ZhbHVlcyJdCiAgICAgICAgICAgIGFsbF9yZWNzOiBsaXN0W2RpY3RdID0gW10KICAgICAgICAgICAgZm9yIHNuIGluIHN0b3JlX251bXM6CiAgICAgICAgICAgICAgICByb3cgPSBzYW1wbGVfZGZbc2FtcGxlX2RmWyJTdG9yZV9OdW1iZXIiXSA9PSBzbl0KICAgICAgICAgICAgICAgIHJvdyA9IHJvdy5pbG9jWzBdIGlmIGxlbihyb3cpIGVsc2Ugc2FtcGxlX2RmLmlsb2NbMF0KICAgICAgICAgICAgICAgIGZvciBpdCBpbiBpdGVtczoKICAgICAgICAgICAgICAgICAgICBhbGxfcmVjcy5hcHBlbmQoewogICAgICAgICAgICAgICAgICAgICAgICAqKnsKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJTdG9yZV9OdW1iZXIiOiAgICBzbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJTdG9yZV9CYW5uZXIiOiAgICBzdHIocm93LmdldCgiU3RvcmVfQmFubmVyIiwgICAgIkhpbGwgQ291bnRyeSBHcm9jZXIiKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiU3RvcmVfUmVnaW9uIjogICAgc3RyKHJvdy5nZXQoIlN0b3JlX1JlZ2lvbiIsICAgICJTb3V0aCAtIFRleGFzIENlbnRyYWwiKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiU3RvcmVfU3FfRnQiOiAgICAgZmxvYXQocm93LmdldCgiU3RvcmVfU3FfRnQiLCAgICAgNDIwMDApKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJTdG9yZV9PcGVuX1llYXIiOiBpbnQocm93LmdldCgiU3RvcmVfT3Blbl9ZZWFyIiwgMjAxMSkpLAogICAgICAgICAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgICAgICAgICAqKml0LAogICAgICAgICAgICAgICAgICAgIH0pCiAgICAgICAgICAgIHJlcyA9IHByZWRpY3RfYmF0Y2hfYXBpKGFsbF9yZWNzLCBtaWQpCiAgICAgICAgICAgIG5fcGVyX3N0b3JlID0gbGVuKGl0ZW1zKQogICAgICAgICAgICB0b3RhbHMgPSB7CiAgICAgICAgICAgICAgICBzbjogc3VtKAogICAgICAgICAgICAgICAgICAgIHBbInByZWRpY3Rpb24iXQogICAgICAgICAgICAgICAgICAgIGZvciBwIGluIHJlc1sicHJlZGljdGlvbnMiXVtqICogbl9wZXJfc3RvcmU6KGogKyAxKSAqIG5fcGVyX3N0b3JlXQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZm9yIGosIHNuIGluIGVudW1lcmF0ZShzdG9yZV9udW1zKQogICAgICAgICAgICB9CiAgICAgICAgICAgIGRmX3N0b3JlcyA9IHBkLkRhdGFGcmFtZSgKICAgICAgICAgICAgICAgIHNvcnRlZCgKICAgICAgICAgICAgICAgICAgICBbeyJTdG9yZSI6IHMsICJQcmVkaWN0ZWQgdG90YWwiOiBmIiR7djosLjJmfSJ9IGZvciBzLCB2IGluIHRvdGFscy5pdGVtcygpXSwKICAgICAgICAgICAgICAgICAgICBrZXk9bGFtYmRhIHI6IC1mbG9hdChyWyJQcmVkaWN0ZWQgdG90YWwiXS5yZXBsYWNlKCIkIiwgIiIpLnJlcGxhY2UoIiwiLCAiIikpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCiAgICAgICAgICAgIHN0LmRhdGFmcmFtZShkZl9zdG9yZXMsIGhpZGVfaW5kZXg9VHJ1ZSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgc3QuZXJyb3Ioc3RyKGV4YykpCgojIEZvb3Rlcgptb2RlbF9zdW1tYXJ5ID0gIiB8ICIuam9pbigKICAgIGYie21bJ21vZGVsX2ZhbWlseSddfSBSwrI9e21bJ3Rlc3RfcjInXTouNGZ9IE1BUEU9e21bJ3Rlc3RfbWFwZV9wY3QnXTouMWZ9JSIKICAgIGZvciBtIGluIG1vZGVsc19saXN0CikKc3QuY2FwdGlvbihmIk1vZGVsczoge21vZGVsX3N1bW1hcnl9IMK3IDk1JSBzcGxpdC1jb25mb3JtYWwgUEkiKQo="
    (backend_dir / "app.py").write_bytes(base64.b64decode(_APP_B64))
    (frontend_dir / "src" / "streamlit_app.py").write_bytes(base64.b64decode(_STLIT_B64))
    print("Space files: decoded from base64 (Colab path)")

print("  backend:", [f.name for f in backend_dir.iterdir()])
print("  frontend:", [f.name for f in frontend_dir.iterdir()])


Space files: copied from build_state
  backend: ['requirements.txt', 'Dockerfile', 'app.py']
  frontend: ['requirements.txt', 'Dockerfile', 'src']


### Deploy to Hugging Face Spaces

**Process:** Create (or confirm) both Spaces and upload all required files. `exist_ok=True` makes this idempotent — re-running updates the Space rather than failing.

**Outcome:** Both Spaces are building on HF infrastructure.

In [14]:
# ------------------------------
# DEPLOY TO HUGGING FACE SPACES
# ------------------------------
import shutil

_base = pathlib.Path("/content") if os.path.exists("/content") else pathlib.Path("/tmp")
bundle_root  = _base / "model_bundle"
backend_dir  = _base / "hf-backend"
frontend_dir = _base / "hf-frontend"

# Backend
print(f"Creating backend Space: {BACKEND_REPO}")
api.create_repo(repo_id=BACKEND_REPO, repo_type="space", space_sdk="docker",
                exist_ok=True, private=False)
for fname in ("Dockerfile", "requirements.txt", "app.py"):
    print(f"  Uploading {fname}...")
    api.upload_file(path_or_fileobj=str(backend_dir / fname),
                    path_in_repo=fname, repo_id=BACKEND_REPO, repo_type="space")
print("  Uploading training_sample.parquet...")
api.upload_file(path_or_fileobj=str(bundle_root / "training_sample.parquet"),
                path_in_repo="training_sample.parquet",
                repo_id=BACKEND_REPO, repo_type="space")
print("  Uploading model_bundle/ directory...")
api.upload_folder(folder_path=str(bundle_root), path_in_repo="model_bundle",
                  repo_id=BACKEND_REPO, repo_type="space",
                  ignore_patterns=["*.pyc", "__pycache__"])
print(f"Backend → https://huggingface.co/spaces/{BACKEND_REPO}")

# Frontend
print(f"\nCreating frontend Space: {FRONTEND_REPO}")
api.create_repo(repo_id=FRONTEND_REPO, repo_type="space", space_sdk="docker",
                exist_ok=True, private=False)
for fname in ("Dockerfile", "requirements.txt"):
    api.upload_file(path_or_fileobj=str(frontend_dir / fname),
                    path_in_repo=fname, repo_id=FRONTEND_REPO, repo_type="space")
api.upload_file(path_or_fileobj=str(frontend_dir / "src" / "streamlit_app.py"),
                path_in_repo="src/streamlit_app.py",
                repo_id=FRONTEND_REPO, repo_type="space")
print(f"Frontend → https://huggingface.co/spaces/{FRONTEND_REPO}")
print("\nBoth Spaces are building. This typically takes 3–7 minutes.")

Creating backend Space: EvagAIML/hill-country-revenue-forecast-backend


No files have been modified since last commit. Skipping to prevent empty commit.


  Uploading Dockerfile...


No files have been modified since last commit. Skipping to prevent empty commit.


  Uploading requirements.txt...
  Uploading app.py...


...

Backend → https://huggingface.co/spaces/EvagAIML/hill-country-revenue-forecast-backend

Creating frontend Space: EvagAIML/hill-country-revenue-forecast-frontend


No files have been modified since last commit. Skipping to prevent empty commit.


No files have been modified since last commit. Skipping to prevent empty commit.


No files have been modified since last commit. Skipping to prevent empty commit.


Frontend → https://huggingface.co/spaces/EvagAIML/hill-country-revenue-forecast-frontend

Both Spaces are building. This typically takes 3–7 minutes.


### Endpoint Smoke Test and Coverage Check

**Process:** Wait for backend readiness then verify all endpoints: `/health`, `/info`, `/predict` (no model, primary, secondary), `/predict_batch`, `/training_sample`, and frontend root. Compute empirical conformal coverage on 100 held-out test rows for each model.

**Analysis:** Coverage should be in [0.85, 1.00] for both models. Models should produce distinct predictions on the same input. The response's `model` field confirms which model served each request.

**Outcome:** All endpoints verified. Coverage and AC21/AC22 checks reported.

In [15]:
# ------------------------------
# ENDPOINT SMOKE TEST AND COVERAGE CHECK
# ------------------------------
import time, requests

print(f"Waiting for backend at {BACKEND_URL} ...")
for attempt in range(120):
    try:
        r = requests.get(f"{BACKEND_URL}/health", timeout=10)
        if r.status_code == 200:
            print(f"  Backend live after ~{attempt * 5}s")
            break
    except Exception:
        pass
    if attempt % 12 == 0 and attempt > 0:
        print(f"  Still waiting... {attempt * 5}s")
    time.sleep(5)

print("/health:", requests.get(f"{BACKEND_URL}/health").json())

info_r = requests.get(f"{BACKEND_URL}/info").json()
print(f"/info: models={len(info_r['models'])}  ci={info_r['ci_mechanism']}  cal_n={info_r['ci_calibration_n']}")

sample_feat = {f["name"]: (f["allowed_values"][0] if f["type"]=="categorical"
                            else float(f.get("min", 1.0))) for f in info_r["features"]}

# No model field → should default to primary
r0 = requests.post(f"{BACKEND_URL}/predict", json={"features": sample_feat}).json()
print(f"/predict (no model):  {r0}")
assert r0["model"] == "primary", f"Expected 'primary', got {r0['model']}"

r_p = requests.post(f"{BACKEND_URL}/predict", json={"features": sample_feat, "model": "primary"}).json()
r_s = requests.post(f"{BACKEND_URL}/predict", json={"features": sample_feat, "model": "secondary"}).json()
print(f"/predict (primary):   {r_p}")
print(f"/predict (secondary): {r_s}")
assert abs(r_p["prediction"] - r_s["prediction"]) > 0.001, "AC21: predictions must differ"
print(f"AC21 distinct: delta={abs(r_p['prediction']-r_s['prediction']):.4f}  PASS")

batch_r = requests.post(f"{BACKEND_URL}/predict_batch",
    json={"records": [sample_feat]*5, "model": "primary"}).json()
print(f"/predict_batch (primary):   count={batch_r['count']}  model={batch_r['model']}")
batch_s = requests.post(f"{BACKEND_URL}/predict_batch",
    json={"records": [sample_feat]*5, "model": "secondary"}).json()
print(f"/predict_batch (secondary): count={batch_s['count']}  model={batch_s['model']}")

# Conformal coverage on 100 test rows
test_records = X_test.iloc[:100].to_dict(orient="records")
y_test_100   = np.array(y_test.iloc[:100])
for mid in ("primary", "secondary"):
    br = requests.post(f"{BACKEND_URL}/predict_batch",
         json={"records": test_records, "model": mid}, timeout=60).json()
    ci_lows  = np.array([p["ci_low"]  for p in br["predictions"]])
    ci_highs = np.array([p["ci_high"] for p in br["predictions"]])
    cov = float(((y_test_100 >= ci_lows) & (y_test_100 <= ci_highs)).mean())
    status = "PASS" if 0.85 <= cov <= 1.00 else "FAIL"
    print(f"{mid} coverage (n=100): {cov:.3f}  [{status}]")

ts_r = requests.get(f"{BACKEND_URL}/training_sample").json()
print(f"/training_sample: {ts_r['n']} rows")

fe_r = requests.get(f"https://{HF_OWNER.lower()}-hill-country-revenue-forecast-frontend.hf.space", timeout=30)
print(f"Frontend HTTP: {fe_r.status_code}")

Waiting for backend at https://evagaiml-hill-country-revenue-forecast-backend.hf.space ...


  Backend live after ~0s


/health: {'status': 'ok'}


/info: models=2  ci=split_conformal  cal_n=1776


/predict (no model):  {'prediction': 48.81, 'ci_low': 8.81, 'ci_high': 93.54, 'currency': 'USD', 'model': 'primary'}


...

/predict_batch (secondary): count=5  model=secondary


primary coverage (n=100): 0.950  [PASS]


secondary coverage (n=100): 0.950  [PASS]


/training_sample: 1000 rows


Frontend HTTP: 200


## **Expanded Executive Summary**

### TLDR

**HistGradientBoostingRegressor** (hyperparams: {'max_depth': 12}) is the primary deployment model, reaching test R² = 0.8994 and MAE = $12.30 per item-store-week. **XGBRegressor** serves as the secondary (R² = 0.8631), accessible via the Administrator model selector for backtests and A/B comparisons. Both models share the same 1,776-row calibration set, so their confidence intervals are directly comparable. The split-conformal mechanism means either model's 95% interval is calibrated independently of model architecture.

### Full Summary

**Objective:** Build and deploy a weekly revenue forecast model for the Hill Country Grocer chain that predicts `Weekly_Revenue_USD` per item × store combination with a calibrated 95% prediction interval, and expose both the primary and a governance-ready secondary model through a single API and frontend.

**Iterative Development:** The gated pipeline harness swept six candidate algorithms on 3-fold cross-validation of the training set. The top-two by CV R² were tuned independently via GridSearchCV and evaluated on the held-out test set. This protocol produces a ranked pair rather than a single winner, enabling model governance without re-running the full pipeline.

**Performance Analysis:** HistGradientBoostingRegressor achieved test R² = 0.8994, MAPE = 15.2%, MAE = $12.30. XGBRegressor achieved R² = 0.8631, MAPE = 17.7%, MAE = $14.34. The 0.0363 R² gap is consistent with the CV selection ordering. Top features for the primary model: `n__List_Price_USD` / `n__Net_Weight_oz` / `n__Promo_Price_USD`. Top features for secondary: `n__List_Price_USD` / `n__Net_Weight_oz` / `n__Promo_Price_USD`. Pricing (`List_Price_USD`, `Promo_Price_USD`) dominates for both, consistent with revenue being the product of price and units sold.

**Economic Impact:** At $12.30 MAE per item-store-week, a chain running 6 stores × ~200 SKUs per week can expect total weekly forecast error in the range of ~$14,757. Accurate forecasting at this level avoids roughly 10% of that in preventable markdown expense. The promo sensitivity surface in the frontend quantifies discount ROI before committing to a flyer, reducing unnecessary markdowns.

**Deployment Readiness:** Backend runs as a Docker container on Hugging Face Spaces exposing `/health`, `/info`, `/predict`, `/predict_batch`, and `/training_sample`. The frontend's Administrator section provides model governance: the model selector switches which model serves all subsequent predictions and batch uploads, supporting operator-controlled A/B comparisons and rollback without redeployment. Enterprise integration path: call `/predict_batch` with the desired `"model"` field from the ERP for ordered governance. SMB path: category managers use the Streamlit UI at weekly planning meetings.

**Next Steps:** (1) Add a week-of-year column to capture seasonal patterns — the current model is time-blind. (2) Retrain on a rolling 52-week window as live sales data accumulates. (3) Add store-level foot traffic as an external signal — the current features are all item or store attributes; transaction volume is a more direct predictor of weekly revenue.